# Model 2: Two-node SCGE model without transport costs

In [12]:
import numpy as np
from scipy.optimize import root

## Functions

### Production

In [13]:
def output(A, L):
    return A * L

### Prices

In [14]:
def price(w, A):
    return w / A

### Household

In [15]:
def utility(beta, Z):
    return np.sum(beta * np.log(Z), axis=0)

def budget_constraint(p, Z, I):
    return np.sum(p[:, np.newaxis] * Z, axis=0) - I

def income(w, L_bar):
    return w * L_bar

def demand_ij(beta, I, p):
    return (beta * I) / p[:, np.newaxis]

### Market clearing / residuals

In [16]:
def goods_market_clearing(Y, Z):
    return Y - np.sum(Z, axis=1)

def value_market_clearing(w, L_bar, beta):
    I = income(w, L_bar)
    demand = np.sum(beta * I, axis=1)
    supply = w * L_bar
    return supply - demand

## Modelling values

In [17]:
L_bar = np.array([100, 100])
A = np.array([1.0, 1.0])
beta = np.array([[0.6, 0.4], [0.4, 0.6]])

## Benchmark

In [18]:
def system_residuals(w2):
    w = np.array([1.0, w2[0]])
    value_clearing = value_market_clearing(w, L_bar, beta)
    return value_clearing[1]

### Value market clearing

In [19]:
sol = root(system_residuals, [2])
print("Solution for w_2:", sol.x[0])

Solution for w_2: 1.0


In [20]:
value_res = system_residuals(sol.x)
print("Value market residuals:", value_res)

Value market residuals: 0.0


In [21]:
w_sol = np.array([1.0, sol.x[0]])
p_sol = price(w_sol, A)
I_sol = income(w_sol, L_bar)
Z_sol = demand_ij(beta, I_sol, p_sol)
Y_sol = output(A, L_bar)

print("Prices:", p_sol)
print("Wages:", w_sol)
print("Demands:", Z_sol)

Prices: [1. 1.]
Wages: [1. 1.]
Demands: [[60. 40.]
 [40. 60.]]


In [22]:
goods_res = goods_market_clearing(Y_sol, Z_sol)
print("Goods market residuals:", goods_res)

Goods market residuals: [0. 0.]


## 3. Explain economically why wages, rather than quantities of labour, adjust across nodes.

Wages are the ones adjusted across nodes and not quantities of labour becasue labour is immobile across regions and regional labour supply is fixed at $\bar{L}_i$, so the physical output supplied by each node $Y_i = A_i \bar{L}_i$ is constant. Therefore, the only way to align goods demand with supply is by changing the value of what each region produces. Consequently it would increase the prices of products and the income of the habitants of the region. 

## 4. Explain why one nominal variable must be normalized.

Because the model is homogeneous of degree zero in all nominal variables, multiplying all wages $w_i$ and prices $p_i$ by a positive constant leaves real incomes, relative prices, and physical allocations $Z_{ij}$ unchanged. Fixing one nominal variable as the numeraire ($w_1 = 1$) is therefore necessary to anchor the absolute price level and uniquely determine all relative prices in the system. Otherwise the model would have infinite answers.

## 5. Show that one market-clearing condition is redundant once household budgets and all other equilibrium conditions hold.

As each household spends its entire income $\sum_i p_i Z_{ij} = I_j = w_j \bar{L}_j$ and zero profits imply $p_i Y_i = w_i \bar{L}_i$, summing household budget constraints across all nodes yields $\sum_i p_i \sum_j Z_{ij} = \sum_j w_j \bar{L}_j = \sum_i p_i Y_i$. If node 1's goods market clears ($Y_1 = \sum_j Z_{1j}$), the remaining goods market for node 2 ($Y_2 = \sum_j Z_{2j}$) is automatically satisfied and mathematically redundant.